# Stan Multidimensional Gaussian Random Restarts

Runs Stan mean-field ADVI random restarts for the diagonal known-covariance Gaussian example.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REL_DIR = Path("single_MC/multidim_gaussian_knowncovar/random_restarts")
STAN_FILE_NAME = "stan_multidim_gaussian_knowncovar.stan"


def find_notebook_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / REL_DIR]
    candidates.extend(parent for parent in cwd.parents)
    candidates.extend(parent / REL_DIR for parent in cwd.parents)
    for candidate in candidates:
        if (candidate / STAN_FILE_NAME).exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {STAN_FILE_NAME} from {cwd}")


NOTEBOOK_DIR = find_notebook_dir()
os.chdir(NOTEBOOK_DIR)

REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "modulars").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Could not locate repo root containing modulars/")
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

STAN_FILE = NOTEBOOK_DIR / STAN_FILE_NAME
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Stan model: {STAN_FILE}")

if Path(sys.prefix).name != "stan3":
    raise RuntimeError(
        f"This notebook must run in the miniconda environment named stan3; "
        f"current sys.prefix is {sys.prefix!r}."
    )

STAN3_PREFIX = Path(sys.prefix).resolve()
STAN3_CMDSTAN = STAN3_PREFIX / "bin" / "cmdstan"
if not STAN3_CMDSTAN.exists():
    raise FileNotFoundError(f"Expected CmdStan at {STAN3_CMDSTAN}")

os.environ["CMDSTAN"] = str(STAN3_CMDSTAN)
from cmdstanpy import cmdstan_path, set_cmdstan_path
set_cmdstan_path(str(STAN3_CMDSTAN))

print(f"Python executable: {sys.executable}")
print(f"CmdStan path: {cmdstan_path()}")


Notebook dir: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/multidim_gaussian_knowncovar/random_restarts
Stan model: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/multidim_gaussian_knowncovar/random_restarts/stan_multidim_gaussian_knowncovar.stan
Python executable: /Users/madelynandersen/miniconda3/envs/stan3/bin/python
CmdStan path: /Users/madelynandersen/miniconda3/envs/stan3/bin/cmdstan


/Users/madelynandersen/miniconda3/envs/stan3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from modulars.distributions import gaussian_multid, gaussian_multid_posterior

mu_prior = np.zeros(40)
sigma_prior = np.ones(40) * 10.0
mu_like = 2.0
sigma_like = 1.0
n_obs = 3
dim = 40

observed_data = gaussian_multid(
    seed=42,
    n_samples=n_obs,
    mu_like=mu_like,
    scale_like=sigma_like,
    dim=dim,
)
true_mu_post, true_sigma_post = gaussian_multid_posterior(
    observed_data,
    n_samples=n_obs,
    mu_prior=mu_prior,
    scale_prior=sigma_prior,
    scale_like=sigma_like,
    std_devs=True,
    diag=True,
    dim=dim,
)

stan_data = {
    "D": dim,
    "N": n_obs,
    "y": observed_data,
    "mu_prior": mu_prior,
    "sigma_prior": sigma_prior,
    "sigma_like": sigma_like,
}
PARAM_COLUMNS = [f"mu[{i}]" for i in range(1, dim + 1)]
LOG_COLUMNS = []


In [3]:
from pathlib import Path

RUN_MODE = os.environ.get("SIMPLEVI_STAN_RUN_MODE", "full")  # "quick" or "full"
RUN_CONFIGS = {
    "quick": {"max_iters": 20, "n_restarts": 1, "track_every": 10, "parallel": False, "max_workers": None, "keep_outputs": True},
    "full": {"max_iters": 300_000, "n_restarts": 20, "track_every": 10, "parallel": True, "max_workers": None, "keep_outputs": False},
}

run_config = RUN_CONFIGS[RUN_MODE]
max_iters = int(os.environ.get("SIMPLEVI_STAN_MAX_ITERS", run_config["max_iters"]))
n_restarts = int(os.environ.get("SIMPLEVI_STAN_N_RESTARTS", run_config["n_restarts"]))
track_every = int(os.environ.get("SIMPLEVI_STAN_TRACK_EVERY", run_config["track_every"]))
parallel = bool(int(os.environ.get("SIMPLEVI_STAN_PARALLEL", int(run_config["parallel"]))))
max_workers_env = os.environ.get("SIMPLEVI_STAN_MAX_WORKERS")
max_workers = int(max_workers_env) if max_workers_env else run_config["max_workers"]
keep_outputs = bool(int(os.environ.get("SIMPLEVI_STAN_KEEP_OUTPUTS", int(run_config["keep_outputs"]))))

results_dir = Path(os.environ.get("SIMPLEVI_STAN_RESULTS_DIR", "stan_cmdstan_output"))
results_dir.mkdir(exist_ok=True, parents=True)

run_config | {"parallel": parallel, "max_workers": max_workers, "results_dir": str(results_dir)}


{'max_iters': 300000,
 'n_restarts': 20,
 'track_every': 10,
 'parallel': True,
 'max_workers': None,
 'keep_outputs': False,
 'results_dir': 'stan_cmdstan_output'}

In [4]:
from modulars.stan_rr_test import run_stan_random_restarts, stan_result_by_scenario, FOUR_WAY_STAN_SCENARIOS

TOL_TARGET_ITERS = 150_000
five_way_scenarios = [dict(s) for s in FOUR_WAY_STAN_SCENARIOS]
for _scenario in five_way_scenarios:
    if _scenario["name"] == "grad100_tol_rel_obj1e-6":
        _scenario["max_iters_scale"] = TOL_TARGET_ITERS / max_iters
five_way_scenarios.append({
    "name": "grad100_tol_rel_obj1e-6_eta0.01",
    "mc_label": "100_tol1e-6_eta0.01",
    "cmd_args": ("grad_samples=100", "adapt", "engaged=0", "eta=0.01"),
    "tol_rel_obj": 1e-6,
    "max_iters_scale": TOL_TARGET_ITERS / max_iters,
    "track_every_scale": 10.0,
})

five_way_scenarios.append({
    "name": "tol_1e-6_default_rest",
    "mc_label": "tol_1e-6_default_rest",
    "cmd_args": (),
    "tol_rel_obj": 1e-6,
    "max_iters_scale": 75_000 / max_iters,
    "track_every_scale": 150 / track_every,
})
stan_result = run_stan_random_restarts(
    stan_file=STAN_FILE,
    data=stan_data,
    param_columns=PARAM_COLUMNS,
    log_columns=LOG_COLUMNS,
    output_dir=results_dir / "cmdstan_runs",
    max_iters=max_iters,
    n_restarts=n_restarts,
    track_every=track_every,
    seed_offset=0,
    parallel=parallel,
    max_workers=max_workers,
    keep_outputs=keep_outputs,
    refresh=0,
    scenarios=five_way_scenarios,
)
if stan_result["failures"]:
    print(f"Stan failures: {len(stan_result['failures'])}; see {results_dir / 'cmdstan_runs' / 'stan_run_failures.csv'}")


13:19:11 - cmdstanpy - INFO - compiling stan file /var/folders/k_/lm3f3ty933bgx8bfm0dsx7cr0000gn/T/tmpuekbsen1/tmp7d20_7x1.stan to exe file /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/multidim_gaussian_knowncovar/random_restarts/stan_multidim_gaussian_knowncovar


13:19:17 - cmdstanpy - INFO - compiled model executable: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/multidim_gaussian_knowncovar/random_restarts/stan_multidim_gaussian_knowncovar


Stan restart/scenario tasks:   0%|          | 0/100 [00:00<?, ?it/s]

Stan restart/scenario tasks:   0%|          | 0/100 [00:00<?, ?it/s]

Stan restart/scenario tasks:   1%|          | 1/100 [01:53<3:06:29, 113.03s/it]

...

Stan restart/scenario tasks:  98%|█████████▊| 98/100 [17:09:59<04:55, 147.58s/it]

Stan restart/scenario tasks:  99%|█████████▉| 99/100 [17:11:39<02:15, 135.32s/it]

Stan restart/scenario tasks: 100%|██████████| 100/100 [20:50:01<00:00, 3662.34s/it]

Stan restart/scenario tasks: 100%|██████████| 100/100 [20:50:01<00:00, 750.02s/it] 

Stan restart/scenario tasks:   0%|          | 0/100 [20:50:02<?, ?it/s]

In [5]:
import pickle

STAN_RESULT_PKL = results_dir / "stan_result.pkl"
with open(STAN_RESULT_PKL, "wb") as f:
    pickle.dump(stan_result, f)
print(f"Saved Stan results to {STAN_RESULT_PKL}")

Saved Stan results to stan_cmdstan_output/stan_result.pkl


In [5]:
import pickle

STAN_RESULT_PKL = results_dir / "stan_result.pkl"
with open(STAN_RESULT_PKL, "rb") as f:
    stan_result = pickle.load(f)
if stan_result["failures"]:
    print(f"Stan failures: {len(stan_result['failures'])}; see {results_dir / 'cmdstan_runs' / 'stan_run_failures.csv'}")
print(f"Loaded Stan results from {STAN_RESULT_PKL}")

Loaded Stan results from stan_cmdstan_output/stan_result.pkl


In [6]:
by_scenario = stan_result_by_scenario(stan_result)

default_means, default_stds, default_iterations = by_scenario["default"]
grad100_means, grad100_stds, grad100_iterations = by_scenario["grad100_default_rest"]
adapt_off_means, adapt_off_stds, adapt_off_iterations = by_scenario["grad100_adapt_off"]
tol_means, tol_stds, tol_iterations = by_scenario["grad100_tol_rel_obj1e-6"]
eta_means, eta_stds, eta_iterations = by_scenario["grad100_tol_rel_obj1e-6_eta0.01"]

import numpy as np
from modulars.plot_rr import plot_some_dims_multid

which_dims = [0, 1, 2]

# "matplotlib" (default, static, savable via plt.savefig) or "plotly"
# (interactive: zoom/pan/hover, shown inline). Flip this and rerun the
# plotting cell below to switch how every plot in this notebook renders.
PLOT_BACKEND = "plotly"


In [7]:
def downsample_and_extend(arr, factor, target_len):
    """Downsample along the last (iteration) axis by `factor`, then hold
    the last value flat out to `target_len` points -- a valid projection
    once a run has converged and stopped changing, used here to compare
    the (short) default run against the (long) tolerance-ablation run on
    the latter's own iteration grid.
    """
    downsampled = arr[..., ::factor]
    n_have = downsampled.shape[-1]
    if n_have < target_len:
        pad = np.repeat(downsampled[..., -1:], target_len - n_have, axis=-1)
        downsampled = np.concatenate([downsampled, pad], axis=-1)
    return downsampled


# non-tolerance comparisons: these three scenarios share the default
# scenario's own iteration grid (all three use the same max_iters/track_every).
# each call also plots the default scenario's own per-dim mean-band, so the
# "default" band appears once per comparison (harmless repetition).
for name, means, stds, scen_iterations in (
    ("grad100_default_rest", grad100_means, grad100_stds, grad100_iterations),
    ("grad100_adapt_off", adapt_off_means, adapt_off_stds, adapt_off_iterations),
):
    plot_some_dims_multid(
        default_means, default_stds, means, stds,
        true_mu_post, true_sigma_post, "mu", dim,
        which_dims=which_dims, k=3, label_prefix=f"Stan {name} ",
        x=scen_iterations, backend=PLOT_BACKEND,
    )

# tolerance comparison: the tolerance scenario runs 2x longer (and is tracked
# 10x coarser) than the default run, so we downsample the default run's
# trajectory by 10x to match the tolerance scenario's tracking step, then
# project its last (converged) value flat out to match tol_iterations'
# length -- it does not change once converged, so this is a valid stand-in
# for actually rerunning the default scenario over the longer window.
projected_default_means = downsample_and_extend(default_means, 10, len(tol_iterations))
projected_default_stds = downsample_and_extend(default_stds, 10, len(tol_iterations))
plot_some_dims_multid(
    projected_default_means, projected_default_stds, tol_means, tol_stds,
    true_mu_post, true_sigma_post, "mu", dim,
    which_dims=which_dims, k=3, label_prefix="Stan grad100_tol_rel_obj1e-6 ",
    x=tol_iterations, backend=PLOT_BACKEND,
)

projected_default_means_eta = downsample_and_extend(default_means, 10, len(eta_iterations))
projected_default_stds_eta = downsample_and_extend(default_stds, 10, len(eta_iterations))
plot_some_dims_multid(
    projected_default_means_eta, projected_default_stds_eta, eta_means, eta_stds,
    true_mu_post, true_sigma_post, "mu", dim,
    which_dims=which_dims, k=3, label_prefix="Stan grad100_tol_rel_obj1e-6_eta0.01 ",
    x=eta_iterations, backend=PLOT_BACKEND,
)


In [1]:
import pickle
import sys
from pathlib import Path
import numpy as np

REPO_ROOT = Path("/Users/madelynandersen/Documents/VB Bakeoff/simpleVI")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.plot_rr import plot_mean_band_rrs_1d
from modulars.distributions import gaussian_multid, gaussian_multid_posterior

NB_DIR = REPO_ROOT / "single_MC/multidim_gaussian_knowncovar/random_restarts"

mu_prior = np.zeros(40)
sigma_prior = np.ones(40) * 10.0
mu_like = 2.0
sigma_like = 1.0
n_obs = 3
dim = 40
which_dims = [0, 1, 2]

observed_data = gaussian_multid(seed=42, n_samples=n_obs, mu_like=mu_like, scale_like=sigma_like, dim=dim)
true_mu_post, true_sigma_post = gaussian_multid_posterior(
    observed_data, n_samples=n_obs, mu_prior=mu_prior, scale_prior=sigma_prior,
    scale_like=sigma_like, std_devs=True, diag=True, dim=dim,
)
best_mus = np.asarray(true_mu_post, dtype=float)
best_stds = np.asarray(true_sigma_post, dtype=float)

with open(NB_DIR / "stan_cmdstan_output/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
tol_means, tol_stds, tol_iterations = _by_scenario["grad100_tol_rel_obj1e-6"]

with open(NB_DIR / "tol_1e-6_default_rest_result.pkl", "rb") as f:
    _td_result = pickle.load(f)
_td_by_scenario = stan_result_by_scenario(_td_result)
td_means, td_stds, td_iterations = _td_by_scenario["tol_1e-6_default_rest"]

n_td_failures = len(_td_result["failures"])
print(f"tol_1e-6_default_rest: {td_means.shape[0]} restarts, {len(td_iterations)} tracked points, "
      f"final iter={int(td_iterations[-1])}")
print(f"failures: {n_td_failures}")

tol_1e-6_default_rest: 10 restarts, 501 tracked points, final iter=75000
failures: 0


In [1]:
for d in which_dims:
    plot_mean_band_rrs_1d(
        td_means[:, :, d], td_stds[:, :, d], best_mus[d], best_stds[d],
        td_iterations, f"mu_dim{d}", "tol_1e-6_default_rest",
        label_prefix="Stan tol_1e-6_default_rest ", backend="plotly",
    )

tol_1e-6_default_rest: 10 restarts, 501 tracked points, final iter=75000
failures: 0


In [1]:
import pickle
import sys
from pathlib import Path
import numpy as np

REPO_ROOT = Path("/Users/madelynandersen/Documents/VB Bakeoff/simpleVI")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.plot_rr import plot_a_few_trajectories_1d
from modulars.distributions import gaussian_multid, gaussian_multid_posterior

NB_DIR = REPO_ROOT / "single_MC/multidim_gaussian_knowncovar/random_restarts"

mu_prior = np.zeros(40)
sigma_prior = np.ones(40) * 10.0
mu_like = 2.0
sigma_like = 1.0
n_obs = 3
dim = 40
which_dims = [0, 1, 2]

observed_data = gaussian_multid(seed=42, n_samples=n_obs, mu_like=mu_like, scale_like=sigma_like, dim=dim)
true_mu_post, true_sigma_post = gaussian_multid_posterior(
    observed_data, n_samples=n_obs, mu_prior=mu_prior, scale_prior=sigma_prior,
    scale_like=sigma_like, std_devs=True, diag=True, dim=dim,
)
best_mus = np.asarray(true_mu_post, dtype=float)
best_stds = np.asarray(true_sigma_post, dtype=float)

with open(NB_DIR / "stan_cmdstan_output/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
default_means, default_stds, default_iterations = _by_scenario["default"]

with open(NB_DIR / "tol_1e-6_default_rest_result.pkl", "rb") as f:
    _td_result = pickle.load(f)
_td_by_scenario = stan_result_by_scenario(_td_result)
td_means, td_stds, td_iterations = _td_by_scenario["tol_1e-6_default_rest"]

def downsample_and_extend(arr, factor, target_len):
    downsampled = arr[..., ::factor]
    n_have = downsampled.shape[-1]
    if n_have < target_len:
        pad = np.repeat(downsampled[..., -1:], target_len - n_have, axis=-1)
        downsampled = np.concatenate([downsampled, pad], axis=-1)
    return downsampled

projected_default_means = downsample_and_extend(default_means, 15, len(td_iterations))
projected_default_stds = downsample_and_extend(default_stds, 15, len(td_iterations))
N_shared = min(default_means.shape[0], td_means.shape[0])
for d in which_dims:
    plot_a_few_trajectories_1d(
        [projected_default_means[:, :, d], projected_default_stds[:, :, d]],
        [td_means[:, :, d], td_stds[:, :, d]],
        best_mus[d], best_stds[d], f"mu_dim{d}", k=2, N=N_shared,
        label_prefix="Stan tol_1e-6_default_rest ", x=td_iterations, backend="plotly",
    )

In [1]:
import pickle
import sys
from pathlib import Path
import numpy as np

REPO_ROOT = Path("/Users/madelynandersen/Documents/VB Bakeoff/simpleVI")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.distributions import gaussian_multid, gaussian_multid_posterior

NB_DIR = REPO_ROOT / "single_MC/multidim_gaussian_knowncovar/random_restarts"

mu_prior = np.zeros(40)
sigma_prior = np.ones(40) * 10.0
mu_like = 2.0
sigma_like = 1.0
n_obs = 3
dim = 40
which_dims = [0, 1, 2]

observed_data = gaussian_multid(seed=42, n_samples=n_obs, mu_like=mu_like, scale_like=sigma_like, dim=dim)
true_mu_post, true_sigma_post = gaussian_multid_posterior(
    observed_data, n_samples=n_obs, mu_prior=mu_prior, scale_prior=sigma_prior,
    scale_like=sigma_like, std_devs=True, diag=True, dim=dim,
)
best_mus_full = np.asarray(true_mu_post, dtype=float)
best_stds_full = np.asarray(true_sigma_post, dtype=float)

with open(NB_DIR / "stan_cmdstan_output/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
tol_means_full, tol_stds_full, tol_iterations = _by_scenario["grad100_tol_rel_obj1e-6"]

with open(NB_DIR / "tol_1e-6_default_rest_result.pkl", "rb") as f:
    _td_result = pickle.load(f)
_td_by_scenario = stan_result_by_scenario(_td_result)
td_means_full, td_stds_full, td_iterations = _td_by_scenario["tol_1e-6_default_rest"]

# select the same 3 representative dims used elsewhere in this notebook
tol_means = tol_means_full[:, :, which_dims]
tol_stds = tol_stds_full[:, :, which_dims]
td_means = td_means_full[:, :, which_dims]
td_stds = td_stds_full[:, :, which_dims]
best_mean_selected = best_mus_full[which_dims]
best_std_selected = best_stds_full[which_dims]

import plotly.graph_objects as go
from plotly.subplots import make_subplots

row_labels = [f"mu_dim{d}" for d in which_dims]
n_rows = len(which_dims)
best_mean_arr = best_mean_selected
best_std_arr = best_std_selected

subplot_titles = []
for lab in row_labels:
    subplot_titles.append(f"{lab}: posterior mean")
    subplot_titles.append(f"{lab}: posterior std")

fig = make_subplots(rows=n_rows, cols=2, vertical_spacing=0.4 / n_rows, horizontal_spacing=0.12,
                     subplot_titles=subplot_titles)

for row_idx in range(n_rows):
    r = row_idx + 1
    tol_m = tol_means
    td_m = td_means
    show_legend = row_idx == 0
    fig.add_trace(go.Scatter(x=tol_iterations, y=np.nanmean(tol_means[:, :, row_idx], axis=0), mode="lines",
                              line=dict(color="rgb(148,103,189)", width=2), opacity=0.65,
                              name="grad100_tol_rel_obj1e-6", legendgroup="tol", showlegend=show_legend), row=r, col=1)
    fig.add_trace(go.Scatter(x=td_iterations, y=np.nanmean(td_means[:, :, row_idx], axis=0), mode="lines",
                              line=dict(color="rgb(31,119,180)", width=2), opacity=0.65,
                              name="tol_1e-6_default_rest", legendgroup="default_rest", showlegend=show_legend), row=r, col=1)
    fig.add_hline(y=best_mean_arr[row_idx], line=dict(color="red", dash="dash"), row=r, col=1)

    fig.add_trace(go.Scatter(x=tol_iterations, y=np.nanmean(tol_stds[:, :, row_idx], axis=0), mode="lines",
                              line=dict(color="rgb(148,103,189)", width=2), opacity=0.65,
                              name="grad100_tol_rel_obj1e-6", legendgroup="tol", showlegend=False), row=r, col=2)
    fig.add_trace(go.Scatter(x=td_iterations, y=np.nanmean(td_stds[:, :, row_idx], axis=0), mode="lines",
                              line=dict(color="rgb(31,119,180)", width=2), opacity=0.65,
                              name="tol_1e-6_default_rest", legendgroup="default_rest", showlegend=False), row=r, col=2)
    fig.add_hline(y=best_std_arr[row_idx], line=dict(color="red", dash="dash"), row=r, col=2)

fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="red", dash="dash"),
                          name="best variational approximation"), row=1, col=1)

for r in range(1, n_rows + 1):
    fig.update_xaxes(title_text="iteration", range=[0, 75_000], row=r, col=1)
    fig.update_xaxes(title_text="iteration", range=[0, 75_000], row=r, col=2)

fig.update_layout(height=300 * n_rows, width=1000, title_text="Stan tol_1e-6_default_rest vs grad100_tol_rel_obj1e-6 (dims 0-2)")
fig.show()

In [1]:
CHECKPOINTS = [50_000, 100_000, 500_000, 1_000_000]

with open(NB_DIR / "default_checkpoint_results.pkl", "rb") as f:
    _cp_results = pickle.load(f)

def pct_error(setup, best):
    return (setup - best) / best * 100.0

def zscore(setup_mean, best_mean, best_std):
    return (setup_mean - best_mean) / best_std

DIM = 0  # just mu[1] (0-indexed dim 0)
label = "mu[1]"
best_mean_d = best_mus[DIM]
best_std_d = best_stds[DIM]

print(f"===== multid_gaussian ({label} only): percent-error / z-score vs true posterior =====")
print(f"reference (true) mean: {best_mean_d:.4f}")
print(f"reference (true) std : {best_std_d:.4f}\n")

rows = []
for cp in CHECKPOINTS:
    r = _cp_results[cp]
    m = np.asarray(r["means_by_scenario"]["default_single"][0, 0, :], dtype=float)[DIM]
    s = np.asarray(r["stds_by_scenario"]["default_single"][0, 0, :], dtype=float)[DIM]
    rows.append((f"default, single restart to {cp}", m, s))

for row_label, scen_name in [
    ("grad100_default_rest (final)", "grad100_default_rest"),
    ("grad100_adapt_off (final)", "grad100_adapt_off"),
    ("grad100_tol_rel_obj1e-6 (final)", "grad100_tol_rel_obj1e-6"),
    ("grad100_tol_rel_obj1e-6_eta0.01 (final)", "grad100_tol_rel_obj1e-6_eta0.01"),
    ("tol_1e-6_default_rest (final)", "tol_1e-6_default_rest"),
]:
    if scen_name == "tol_1e-6_default_rest":
        means, stds, iters = td_means, td_stds, td_iterations
    else:
        means, stds, iters = _by_scenario[scen_name]
    m = np.nanmean(means[:, -1, DIM])
    s = np.nanmean(stds[:, -1, DIM])
    rows.append((f"{row_label} (iter={int(iters[-1])})", m, s))

for row_name, m, s in rows:
    pe_mean = pct_error(m, best_mean_d)
    pe_std = pct_error(s, best_std_d)
    z = zscore(m, best_mean_d, best_std_d)
    print(f"  {row_name:45s} {label:10s}  pct_err_mean={pe_mean:9.3f}%  pct_err_std={pe_std:9.3f}%  z={z:7.3f}")

===== multid_gaussian (mu[1] only): percent-error / z-score vs true posterior =====
reference (true) mean: 2.3307
reference (true) std : 0.5764

  default, single restart to 50000              mu[1]       pct_err_mean=    0.073%  pct_err_std=    1.344%  z=  0.003
  default, single restart to 100000             mu[1]       pct_err_mean=    0.073%  pct_err_std=    1.344%  z=  0.003
  default, single restart to 500000             mu[1]       pct_err_mean=    0.073%  pct_err_std=    1.344%  z=  0.003
  default, single restart to 1000000            mu[1]       pct_err_mean=    0.073%  pct_err_std=    1.344%  z=  0.003
  grad100_default_rest (final) (iter=30000)     mu[1]       pct_err_mean=   -0.255%  pct_err_std=   -0.832%  z= -0.010
  grad100_adapt_off (final) (iter=30000)        mu[1]       pct_err_mean=   -0.145%  pct_err_std=   -0.088%  z= -0.006
  grad100_tol_rel_obj1e-6 (final) (iter=150000) mu[1]       pct_err_mean=    0.080%  pct_err_std=   -0.145%  z=  0.003
  grad100_tol_rel_obj1

In [1]:
import pickle
import sys
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path("/Users/madelynandersen/Documents/VB Bakeoff/simpleVI")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.distributions import gaussian_multid, gaussian_multid_posterior

NB_DIR = REPO_ROOT / "single_MC/multidim_gaussian_knowncovar/random_restarts"

mu_prior = np.zeros(40)
sigma_prior = np.ones(40) * 10.0
mu_like = 2.0
sigma_like = 1.0
n_obs = 3
dim = 40
which_dims = [0, 1, 2]

observed_data = gaussian_multid(seed=42, n_samples=n_obs, mu_like=mu_like, scale_like=sigma_like, dim=dim)
true_mu_post, true_sigma_post = gaussian_multid_posterior(
    observed_data, n_samples=n_obs, mu_prior=mu_prior, scale_prior=sigma_prior,
    scale_like=sigma_like, std_devs=True, diag=True, dim=dim,
)
best_mus = np.asarray(true_mu_post, dtype=float)
best_stds = np.asarray(true_sigma_post, dtype=float)

with open(NB_DIR / "stan_cmdstan_output/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
default_means, default_stds, default_iterations = _by_scenario["default"]
default_trace_mean = default_means[0]
default_trace_std = default_stds[0]

CHECKPOINTS = [50_000, 100_000, 500_000, 1_000_000]
with open(NB_DIR / "default_checkpoint_results.pkl", "rb") as f:
    _cp_results = pickle.load(f)
cp_means_full = np.array([np.asarray(_cp_results[cp]["means_by_scenario"]["default_single"][0, 0, :], dtype=float) for cp in CHECKPOINTS])
cp_stds_full = np.array([np.asarray(_cp_results[cp]["stds_by_scenario"]["default_single"][0, 0, :], dtype=float) for cp in CHECKPOINTS])
cp_iters = np.array(CHECKPOINTS)

row_labels = [f"mu_dim{d}" for d in which_dims]
n_rows = len(which_dims)
subplot_titles = []
for lab in row_labels:
    subplot_titles.append(f"{lab}: posterior mean")
    subplot_titles.append(f"{lab}: posterior std")

fig = make_subplots(rows=n_rows, cols=2, vertical_spacing=0.4 / n_rows, horizontal_spacing=0.12,
                     subplot_titles=subplot_titles)

for row_idx, d in enumerate(which_dims):
    r = row_idx + 1
    show_legend = row_idx == 0
    fig.add_trace(go.Scatter(x=default_iterations, y=default_trace_mean[:, d], mode="lines",
                              line=dict(color="rgb(31,119,180)", width=1.5),
                              name="Single restart (same seed)", legendgroup="line", showlegend=show_legend), row=r, col=1)
    fig.add_trace(go.Scatter(x=cp_iters, y=cp_means_full[:, d], mode="markers", marker=dict(color="orange", size=9),
                              name="Single extended run example", legendgroup="dots", showlegend=show_legend), row=r, col=1)
    fig.add_hline(y=best_mus[d], line=dict(color="red", dash="dash"), row=r, col=1)

    fig.add_trace(go.Scatter(x=default_iterations, y=default_trace_std[:, d], mode="lines",
                              line=dict(color="rgb(31,119,180)", width=1.5),
                              name="Single restart (same seed)", legendgroup="line", showlegend=False), row=r, col=2)
    fig.add_trace(go.Scatter(x=cp_iters, y=cp_stds_full[:, d], mode="markers", marker=dict(color="orange", size=9),
                              name="Single extended run example", legendgroup="dots", showlegend=False), row=r, col=2)
    fig.add_hline(y=best_stds[d], line=dict(color="red", dash="dash"), row=r, col=2)

fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="red", dash="dash"),
                          name="true posterior"), row=1, col=1)

for r in range(1, n_rows + 1):
    fig.update_xaxes(type="log", title_text="iteration (log scale)", row=r, col=1)
    fig.update_xaxes(type="log", title_text="iteration (log scale)", row=r, col=2)

fig.update_layout(height=300 * n_rows, width=1000, title_text="multid_gaussian (dims 0-2) — posterior mean/std, default settings out to long iteration horizons")
fig.show()